# Train Odoo Addon Migrator Brain (Community + authorized Enterprise)

This notebook builds two source-free runtime artifacts for Odoo 14 through 19:

1. `migration_brain_community_14_19.omb` from exact verified Odoo Community commits.
2. `enterprise_overlay_14_19.omb` from the authorized Enterprise trees in the supplied Drive folder.

The Enterprise folders are read-only training input. The resulting `.omb` contains derived mappings/model parameters only, is leakage-audited, and is marked `local_authorized_use_only`. Confirm that your Odoo Enterprise agreement and company policy allow processing the source in Google Colab/Drive. Do not distribute the Enterprise overlay unless your license explicitly permits it.

A GPU is not required: the current semantic ranker and source indexer are CPU/RAM intensive. Choose a **High-RAM CPU runtime** when available. Training can take hours. Progress between 50% and 62% reports Git-history/dataset stages rather than appearing frozen.

## One-time Drive preparation

Do **not** train from thousands of individual files through the mounted Drive filesystem; that can take more than a day. On the Windows machine that holds the authorized source, first run:

```powershell
python scripts/package_enterprise_for_colab.py --source D:\odoo\SourceCode\OdooEnterprise --output D:\odoo\OdooEnterpriseArchives
```

Upload the resulting six `odoo-enterprise-14.0.zip` through `odoo-enterprise-19.0.zip` files to `My Drive/OdooEnterpriseArchives`. Colab then transfers six sequential archive files and extracts them on its local SSD. The original Enterprise trees remain untouched.

The shared-folder shortcut remains available as a slow fallback: open the shared [Enterprise source folder](https://drive.google.com/drive/folders/1yKOhhT2-Quhy_Va42a4-rUkdsoB7FtYt?usp=sharing), choose **Organize → Add shortcut**, place it in **My Drive**, and name it `OdooEnterprise`. Set `USE_ENTERPRISE_ARCHIVES = False` only if archive preparation is impossible.

Google Colab's Drive mount does not reliably expose items that are only under *Shared with me*; the shortcut makes the authorized folder visible without duplicating it.

The Odoo Addon Migrator GitHub repository is private. In Colab, open the **Secrets** panel (key icon), add a secret named `GITHUB_TOKEN`, paste a fine-grained GitHub token with read-only **Contents** access to this repository, and enable **Notebook access**. The token is passed to Git only through the clone process environment; it is not written into this notebook or the repository URL.

In [ ]:
# Configuration
ENTERPRISE_ARCHIVE_ROOT = '/content/drive/MyDrive/OdooEnterpriseArchives'
ENTERPRISE_DRIVE_ROOT = '/content/drive/MyDrive/OdooEnterprise'  # slow fallback only
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/OdooAddonMigratorBrain'
WORK_DIR = '/content/oam-training'  # fast, disposable Colab local disk

REPOSITORY_URL = 'https://github.com/UsamaFathi/Odoo-Addon-Migrator.git'
PROJECT_COMMIT = 'e32b5c7e746a97b7a038617e40ca6ffae25878ed'
SOURCE_VERSION = 14
TARGET_VERSION = 19

# Strongly recommended: transfer one ZIP per version instead of ~164,000 Drive files.
USE_ENTERPRISE_ARCHIVES = True
# Used only by the slow shared-folder fallback.
STAGE_ENTERPRISE_LOCALLY = True
REUSE_EXISTING_COMMUNITY_BRAIN = True
REUSE_EXISTING_ENTERPRISE_OVERLAY = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
enterprise_archive_root = Path(ENTERPRISE_ARCHIVE_ROOT)
enterprise_root = Path(ENTERPRISE_DRIVE_ROOT)
if USE_ENTERPRISE_ARCHIVES:
    missing = [str(enterprise_archive_root / f'odoo-enterprise-{version}.0.zip') for version in range(SOURCE_VERSION, TARGET_VERSION + 1) if not (enterprise_archive_root / f'odoo-enterprise-{version}.0.zip').is_file()]
    if missing:
        raise FileNotFoundError('Missing Enterprise archives:\n' + '\n'.join(missing))
    print('All Enterprise archives are available. Original source will not be modified:', enterprise_archive_root)
elif not enterprise_root.is_dir():
    raise FileNotFoundError(f'Enterprise shortcut not found at {enterprise_root}.')
else:
    print('WARNING: direct mounted-folder mode is very slow:', enterprise_root)

In [ ]:
# Install the exact tested trainer implementation.
import base64
import os
import shutil
import subprocess
from pathlib import Path
from google.colab import userdata

repo = Path('/content/Odoo-Addon-Migrator')
if repo.exists():
    shutil.rmtree(repo)
clone_command = ['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(repo)]
clone = subprocess.run(clone_command, text=True, capture_output=True)
if clone.returncode != 0:
    if repo.exists():
        shutil.rmtree(repo)
    try:
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None
    if not github_token:
        raise RuntimeError(
            'The GitHub repository is private. Add a Colab secret named GITHUB_TOKEN, ' 
            'grant it read-only access to UsamaFathi/Odoo-Addon-Migrator, enable Notebook access, ' 
            'then run this cell again. Git said: ' + clone.stderr.strip()
        )
    credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    git_environment = os.environ.copy()
    git_environment.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    clone = subprocess.run(clone_command, text=True, capture_output=True, env=git_environment)
    if clone.returncode != 0:
        raise RuntimeError(
            'Private GitHub clone failed. Confirm the token can read this repository. Git said: ' 
            + clone.stderr.strip()
        )
print('Repository cloned successfully.')
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PROJECT_COMMIT], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)
actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == PROJECT_COMMIT, (actual, PROJECT_COMMIT)
print('Trainer checkout:', actual)

In [ ]:
# Validate all six Enterprise version folders before expensive training starts.
import sys
sys.path.insert(0, str(repo / 'src'))
from odoo_migrator.sources.enterprise import resolve_enterprise_source
from odoo_migrator.sources.registry import VERIFIED_COMMUNITY_COMMITS

enterprise_roots = {}
if USE_ENTERPRISE_ARCHIVES:
    for version in range(SOURCE_VERSION, TARGET_VERSION + 1):
        archive = enterprise_archive_root / f'odoo-enterprise-{version}.0.zip'
        print(f'Odoo {version}: Enterprise archive {archive.name} ({archive.stat().st_size / 1024 / 1024:.1f} MiB); Community {VERIFIED_COMMUNITY_COMMITS[version][:12]}')
else:
    for version in range(SOURCE_VERSION, TARGET_VERSION + 1):
        resolved = resolve_enterprise_source(enterprise_root, version)
        enterprise_roots[version] = resolved.source_root
        print(f'Odoo {version}: Enterprise {resolved.mode} validated; Community {VERIFIED_COMMUNITY_COMMITS[version][:12]}')

In [ ]:
# Build both packs with explicit caches and continuously streamed progress.
import logging
import time
from odoo_migrator.brain.overlay import EnterpriseOverlayTrainer
from odoo_migrator.brain.pack import BrainPack
from odoo_migrator.brain.trainer import BrainTrainer
from odoo_migrator.sources.indexer import SourceIndexer
from odoo_migrator.sources.manager import SourceManager

work_dir = Path(WORK_DIR)
output_dir = Path(OUTPUT_DRIVE_DIR)
work_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler(output_dir / 'migration_brain_training.log')],
    force=True,
)

if USE_ENTERPRISE_ARCHIVES:
    import stat
    import zipfile
    def transfer_archive(source, destination):
        destination.parent.mkdir(parents=True, exist_ok=True)
        total = source.stat().st_size
        if destination.is_file() and destination.stat().st_size == total:
            print(f'Reusing local archive: {destination.name}', flush=True)
            return
        temporary = destination.with_suffix(destination.suffix + '.copying')
        temporary.unlink(missing_ok=True)
        copied = 0
        next_report = 0
        with source.open('rb') as source_stream, temporary.open('wb') as target_stream:
            while True:
                block = source_stream.read(16 * 1024 * 1024)
                if not block:
                    break
                target_stream.write(block)
                copied += len(block)
                percent = int(copied / total * 100)
                if percent >= next_report:
                    print(f'Transfer {source.name}: {percent}% ({copied / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MiB)', flush=True)
                    next_report += 10
        temporary.replace(destination)
    def extract_archive(archive_path, destination, version):
        marker = destination / '.archive-complete'
        identity = f'{archive_path.name}:{archive_path.stat().st_size}'
        if marker.is_file() and marker.read_text() == identity:
            resolved = resolve_enterprise_source(destination, version)
            print(f'Reusing extracted Enterprise Odoo {version}', flush=True)
            return resolved.source_root
        temporary = destination.with_name(destination.name + '.extracting')
        if temporary.exists():
            shutil.rmtree(temporary)
        temporary.mkdir(parents=True)
        try:
            with zipfile.ZipFile(archive_path) as bundle:
                members = bundle.infolist()
                base = temporary.resolve()
                for item in members:
                    target = (temporary / item.filename).resolve()
                    if not target.is_relative_to(base) or stat.S_ISLNK(item.external_attr >> 16):
                        raise ValueError(f'Unsafe archive member: {item.filename}')
                for index, item in enumerate(members, start=1):
                    bundle.extract(item, temporary)
                    if index % 2000 == 0 or index == len(members):
                        print(f'Extract Odoo {version}: {index:,}/{len(members):,} files', flush=True)
            resolved = resolve_enterprise_source(temporary, version)
            marker_in_temporary = temporary / '.archive-complete'
            marker_in_temporary.write_text(identity)
            if destination.exists():
                shutil.rmtree(destination)
            temporary.replace(destination)
            return resolve_enterprise_source(destination, version).source_root
        except BaseException:
            shutil.rmtree(temporary, ignore_errors=True)
            raise
    enterprise_roots = {}
    for version in range(SOURCE_VERSION, TARGET_VERSION + 1):
        drive_archive = enterprise_archive_root / f'odoo-enterprise-{version}.0.zip'
        local_archive = work_dir / 'archives' / drive_archive.name
        print(f'Preparing Enterprise Odoo {version} archive...', flush=True)
        transfer_archive(drive_archive, local_archive)
        enterprise_roots[version] = extract_archive(local_archive, work_dir / 'enterprise' / str(version), version)
elif STAGE_ENTERPRISE_LOCALLY:
    print('WARNING: copying individual Drive files is the slow fallback mode.', flush=True)
    staged_roots = {}
    for version, source_path in enterprise_roots.items():
        destination = work_dir / 'enterprise' / f'{version}.0'
        if destination.exists():
            shutil.rmtree(destination)
        print(f'Staging Enterprise Odoo {version} on Colab local disk (Drive remains read-only)...', flush=True)
        shutil.copytree(source_path, destination, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'))
        staged_roots[version] = destination
    enterprise_roots = staged_roots

class ColabSourceIndexer(SourceIndexer):
    def __init__(self, cache_dir):
        super().__init__()
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    def index(self, root, **kwargs):
        kwargs.setdefault('cache_dir', self.cache_dir)
        return super().index(root, **kwargs)

started = time.perf_counter()
def progress(scope):
    def report(message, percent):
        print(f'{scope} {percent:3d}% | {message} | elapsed {time.perf_counter() - started:.1f}s', flush=True)
    return report

source_manager = SourceManager(cache_root=work_dir / 'cache' / 'sources')
indexer = ColabSourceIndexer(work_dir / 'cache' / 'indexes')
community_path = output_dir / f'migration_brain_community_{SOURCE_VERSION}_{TARGET_VERSION}.omb'
community = BrainPack.load(community_path) if REUSE_EXISTING_COMMUNITY_BRAIN and community_path.is_file() else None
if community is not None:
    assert community.pack_kind == 'community'
    assert (community.source, community.target) == (SOURCE_VERSION, TARGET_VERSION)
    for version in range(SOURCE_VERSION, TARGET_VERSION + 1):
        assert community.source_identities[str(version)]['community_commit'] == VERIFIED_COMMUNITY_COMMITS[version]
    print('Reusing Community Brain:', community.fingerprint)
else:
    community_result = BrainTrainer(source_manager=source_manager, indexer=indexer).build(
        community_path, source=SOURCE_VERSION, target=TARGET_VERSION, progress=progress('COMMUNITY')
    )
    community = BrainPack.load(community_result.output)

overlay_path = output_dir / f'enterprise_overlay_{SOURCE_VERSION}_{TARGET_VERSION}.omb'
overlay = BrainPack.load(overlay_path) if REUSE_EXISTING_ENTERPRISE_OVERLAY and overlay_path.is_file() else None
if overlay is not None:
    assert overlay.pack_kind == 'enterprise_overlay'
    assert overlay.base_fingerprint == community.fingerprint
    print('Reusing Enterprise overlay:', overlay.fingerprint)
else:
    overlay_result = EnterpriseOverlayTrainer(source_manager=source_manager, indexer=indexer).build(
        community, overlay_path, enterprise_roots=enterprise_roots, progress=progress('ENTERPRISE')
    )
    overlay = BrainPack.load(overlay_result.output)
print('Training artifacts are ready in', output_dir)

In [ ]:
# Verify fingerprints/leakage and run a source-free 16 -> 18 migration smoke test.
import hashlib
import json
from datetime import datetime, timezone
from odoo_migrator.brain.runtime import BrainRuntimeMigrator

assert community.distribution_policy == 'public_community'
assert overlay.distribution_policy == 'local_authorized_use_only'
assert overlay.payload['source_leakage_audit']['status'] == 'passed'
assert community.contains_source_code is False and overlay.contains_source_code is False

smoke_root = work_dir / 'source-free-smoke'
if smoke_root.exists():
    shutil.rmtree(smoke_root)
addon = smoke_root / 'custom_addons' / 'brain_smoke_addon'
addon.mkdir(parents=True)
(addon / '__init__.py').write_text('')
(addon / '__manifest__.py').write_text("{'name': 'Brain smoke', 'version': '16.0.1.0.0', 'depends': ['base'], 'installable': True}\n")
input_root = addon.parent
before = SourceIndexer.project_fingerprint(input_root)
smoke_result = BrainRuntimeMigrator(community, overlay=overlay).migrate(
    input_root, smoke_root / 'migrated', source=16, target=18
)
assert SourceIndexer.project_fingerprint(input_root) == before
smoke_metadata = json.loads(smoke_result.metadata_path.read_text())
assert smoke_metadata['source_code_indexed_at_runtime'] is False
assert '18.0.1.0.0' in (smoke_result.output / addon.name / '__manifest__.py').read_text()

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

checksums = {community_path.name: sha256(community_path), overlay_path.name: sha256(overlay_path)}
summary = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'project_commit': actual,
    'verified_community_commits': {str(v): VERIFIED_COMMUNITY_COMMITS[v] for v in range(SOURCE_VERSION, TARGET_VERSION + 1)},
    'community': {'fingerprint': community.fingerprint, 'production_validation': community.training.get('production_validation', {})},
    'enterprise_overlay': {'fingerprint': overlay.fingerprint, 'base_fingerprint': overlay.base_fingerprint, 'production_validation': overlay.training.get('production_validation', {}), 'source_leakage_audit': overlay.payload['source_leakage_audit']},
    'source_free_smoke': {'path': '16_to_17,17_to_18', 'input_unchanged': True, 'source_code_indexed_at_runtime': False, 'validation_state': smoke_result.validation_state},
    'sha256': checksums,
}
(output_dir / 'migration_brain_training_summary.json').write_text(json.dumps(summary, indent=2))
(output_dir / 'SHA256SUMS.txt').write_text(''.join(f'{digest}  {name}\n' for name, digest in checksums.items()))
print(json.dumps(summary, indent=2))
print('\nArtifacts:')
for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size / 1024 / 1024:.2f} MiB')

## Runtime use (no Odoo source required)

After training, normal users need only their custom addons plus the Community Brain and Enterprise overlay:

```bash
odoo-migrator brain migrate CUSTOM_ADDONS OUTPUT \
  --brain migration_brain_community_14_19.omb \
  --overlay enterprise_overlay_14_19.omb --from 16 --to 19
```

The runtime does not index or require Community/Enterprise Odoo source. Automatic changes remain limited to deterministic transformations and high-confidence mappings; ambiguous changes stay in the decision report. Static validation is not runtime compatibility proof.